In [1]:
import torch
import pandas as pd
import numpy as np
from pyvene import embed_to_distrib, top_vals
from pyvene import (
    IntervenableModel,
    VanillaIntervention,
    RepresentationConfig,
    IntervenableConfig,
    ConstantSourceIntervention,
    LocalistRepresentationIntervention
)
from pyvene import create_gpt2
from tqdm import tqdm
from plotnine import *

In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
config, tokenizer, gpt = create_gpt2(name="gpt2-xl")
gpt.to(device)
gpt.eval()

loaded model


GPT2Model(
  (wte): Embedding(50257, 1600)
  (wpe): Embedding(1024, 1600)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-47): 48 x GPT2Block(
      (ln_1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=4800, nx=1600)
        (c_proj): Conv1D(nf=1600, nx=1600)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=6400, nx=1600)
        (c_proj): Conv1D(nf=1600, nx=6400)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
)

In [3]:
class NoiseIntervention(ConstantSourceIntervention, LocalistRepresentationIntervention):
    def __init__(self, embed_dim, **kwargs):
        super().__init__()
        self.interchange_dim = embed_dim
        self.noise_level = 0.13462981581687927
        self.noise = None

    def forward(self, base, source=None, subspaces=None):
        current_len = base.shape[1]

        if self.noise is None or self.noise.shape[1] != current_len:
            rs = np.random.RandomState(1)
            prng = lambda *shape: rs.randn(*shape)
            self.noise = torch.from_numpy(
                prng(1, current_len, self.interchange_dim)
            ).to(base.device)
            
        base[..., : self.interchange_dim] += self.noise * self.noise_level
        return base

In [4]:

examples = [
    {"subject": "The Eiffel Tower", "context": " is located in the city of", "target": " Paris"},
    {"subject": "Barack Obama", "context": " was born in the state of", "target": " Hawaii"},
    {"subject": "The Beatles", "context": " originated from the city of", "target": " Liverpool"},
    {"subject": "Shakespeare", "context": " wrote the play Romeo and", "target": " Juliet"},
    {"subject": "Steve Jobs", "context": " was the founder of", "target": " Apple"}
]

In [5]:
def get_range_of_subject(tokenizer, subject, full_prompt):
    # znajdujemy indeksy tokenów podmiotu
    full_ids = tokenizer.encode(full_prompt)
    subject_ids = tokenizer.encode(subject)
    
    # Szukamy podciągu subject_ids w full_ids
    len_sub = len(subject_ids)
    for i in range(len(full_ids) - len_sub + 1):
        if full_ids[i : i + len_sub] == subject_ids:
            return list(range(i, i + len_sub))
    # Zakładamy, że podmiot jest na początku zdania dla tych przykładów
    return list(range(len(subject_ids)))

In [ ]:
# Funkcja pomocnicza do generowania configu
def restore_corrupted_with_interval_config(layer, stream="mlp_activation", window=10, num_layers=48):
    start = max(0, layer - window // 2)
    end = min(num_layers, layer - (-window // 2))
    config = IntervenableConfig(
        representations=[
            RepresentationConfig(0, "block_input"), # Warstwa do zaszumiania
        ] + [
            RepresentationConfig(i, stream) for i in range(start, end) # Warstwy do przywracania
        ],
        intervention_types=[NoiseIntervention] + [VanillaIntervention] * (end - start),
    )
    return config

In [ ]:
titles = {
    "block_output": "Single restored layer",
    "mlp_activation": "Center of interval of 10 patched MLP layers",
    "attention_output": "Center of interval of 10 patched Attn layers"
}
colors = {"block_output": "Purples", "mlp_activation": "Greens", "attention_output": "Reds"}


for i, ex in enumerate(examples):
    full_prompt = ex["subject"] + ex["context"]
    target_str = ex["target"]
    
    print(f"\n--- Processing Example {i+1}: '{full_prompt}' -> '{target_str}' ---")
    
    base_tensors = tokenizer(full_prompt, return_tensors="pt").to(device)
    target_id = tokenizer.encode(target_str)[0]
    subject_indices = get_range_of_subject(tokenizer, ex["subject"], full_prompt)
    print(f"Subject tokens indices: {subject_indices}")
    
    streams_to_plot = ["mlp_activation"]
    
    for stream in streams_to_plot:
        data = []
        n_layers = gpt.config.n_layer
        n_tokens = base_tensors.input_ids.shape[1]
        
        for layer_i in tqdm(range(n_layers), desc=f"Scanning {stream}"):
            for pos_i in range(n_tokens):
                config = restore_corrupted_with_interval_config(
                    layer_i, stream, 
                    window=1 if stream == "block_output" else 10
                )
                n_restores = len(config.representations) - 1
                
                intervenable = IntervenableModel(config, gpt)
                
                _, counterfactual_outputs = intervenable(
                    base_tensors,
                    [None] + [base_tensors] * n_restores,
                    {
                        "sources->base": (
                            [None] + [[[pos_i]]] * n_restores,
                            # POPRAWIONE ZAGNIEŻDŻENIE:
                            [[subject_indices]] + [[[pos_i]]] * n_restores, 
                        )
                    },
                )
                
                distrib = embed_to_distrib(gpt, counterfactual_outputs.last_hidden_state, logits=False)
                prob = distrib[0][-1][target_id].detach().cpu().item()
                data.append({"layer": layer_i, "pos": pos_i, "prob": prob})
        
        df = pd.DataFrame(data)
        token_labels = [tokenizer.decode([x]) for x in base_tensors.input_ids[0]]
        token_labels = [f"{lbl} ({j})" for j, lbl in enumerate(token_labels)]
        
        plot = (
            ggplot(df, aes(x="layer", y="pos"))
            + geom_tile(aes(fill="prob"))
            + scale_fill_cmap(colors[stream])
            + xlab(f"{titles[stream]} \n Example: {ex['subject']}...")
            + scale_y_reverse(
                limits=(-0.5, n_tokens - 0.5),
                breaks=range(n_tokens),
                labels=token_labels
            )
            + theme(figure_size=(6, 4))
            + ylab("Token Position")
            + theme(axis_text_y=element_text(angle=0, hjust=1))
            + ggtitle(f"Target: {target_str}")
        )
        print(plot)
        ggsave(plot, filename=f"tutorial_with_examples_data/rome_ex_{i}_{stream}.pdf", dpi=200)


--- Processing Example 1: 'The Eiffel Tower is located in the city of' -> ' Paris' ---
Subject tokens indices: [0, 1, 2, 3, 4]


Scanning mlp_activation: 100%|██████████| 48/48 [03:38<00:00,  4.54s/it]
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 4 in image.
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: tutorial_with_examples_data/rome_ex_0_mlp_activation.pdf


<ggplot: (600 x 400)>

--- Processing Example 2: 'Barack Obama was born in the state of' -> ' Hawaii' ---
Subject tokens indices: [0, 1, 2]


Scanning mlp_activation: 100%|██████████| 48/48 [02:59<00:00,  3.74s/it]
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 4 in image.
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: tutorial_with_examples_data/rome_ex_1_mlp_activation.pdf


<ggplot: (600 x 400)>

--- Processing Example 3: 'The Beatles originated from the city of' -> ' Liverpool' ---
Subject tokens indices: [0, 1]


...

Scanning mlp_activation: 100%|██████████| 48/48 [02:20<00:00,  2.92s/it]
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 4 in image.
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: tutorial_with_examples_data/rome_ex_3_mlp_activation.pdf


<ggplot: (600 x 400)>

--- Processing Example 5: 'Steve Jobs was the founder of' -> ' Apple' ---
Subject tokens indices: [0, 1]


Scanning mlp_activation: 100%|██████████| 48/48 [01:58<00:00,  2.47s/it]

<ggplot: (600 x 400)>



/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6 x 4 in image.
/home/administrator/.venv/lib/python3.12/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: tutorial_with_examples_data/rome_ex_4_mlp_activation.pdf


## 1. Na czym polega technika Causal Tracing?
Causal Tracing (Śledzenie przyczynowe) to metoda diagnostyczna, która pozwala odpowiedzieć na pytanie: „Która część sieci neuronowej jest odpowiedzialna za to, że model wie ten konkretny fakt?”.

Technika ta opiera się na Analizie Mediacji Przyczynowej (Causal Mediation Analysis) i przebiega w trzech krokach:

Przebieg Czysty (Clean Run): Uruchamiamy model z normalnym zapytaniem (np. „Steve Jobs był założycielem...”) i zapisujemy wszystkie aktywacje (wartości wewnątrz neuronów). Model poprawnie przewiduje „Apple”.

Przebieg Uszkodzony (Corrupted Run): Do reprezentacji wektorowej podmiotu („Steve Jobs”) dodajemy szum (np. szum Gaussa). To sprawia, że model traci informację o tym, o kogo chodzi, i przewiduje coś innego (błędna odpowiedź). To właśnie robiliśmy w Twoim kodzie, modyfikując klasę NoiseIntervention.

Przebieg Przywrócony (Restored Run): To najważniejszy etap. Podczas gdy model przetwarza „zaszumione” dane, my chirurgicznie kopiujemy (przywracamy) czyste aktywacje z kroku 1 w konkretnym miejscu (np. tylko w 15. warstwie MLP na pozycji 2. tokena).

Jeśli po przywróceniu tego małego fragmentu model nagle „przypomina sobie” poprawną odpowiedź („Apple”), oznacza to, że ten konkretny fragment sieci zawiera kluczową wiedzę.

## 2. Które tokeny podmiotu mają największy wpływ na predykcję?
Z badań przeprowadzonych w artykule (i co widać na wykresach generowanych przez Twój kod) wynika jednoznacznie, że największy wpływ ma ostatni token podmiotu.

Dla „Steve Jobs”: kluczowy jest token „ Jobs”.

Dla „The Eiffel Tower”: kluczowy jest token „ Tower”.

Dlaczego? Modele językowe typu Transformer przetwarzają tekst sekwencyjnie, ale mają mechanizm, który pozwala im agregować informacje. Badacze odkryli, że model „czeka” do końca nazwy podmiotu, aby rozwiązać odniesienie do konkretnego obiektu w świecie rzeczywistym. Dopiero gdy model przetworzy słowo „Tower”, kompletuje w swoich wewnętrznych warstwach reprezentację „Wieży Eiffla” i pobiera z pamięci fakty z nią związane.

## 3. Dlaczego wzorce aktywacji dla attention_output, block_output i mlp_activation różnią się?
Każdy z tych komponentów pełni inną funkcję w architekturze Transformera, co widać na wykresach Causal Tracing:

A. MLP Activation (Aktywacja warstw gęstych) – Magazyn Wiedzy
Wzorzec: Zwykle widzimy silną plamę aktywności w środkowych warstwach (np. warstwy 10–20 w GPT-2 XL) dokładnie na ostatnim tokenie podmiotu.

Funkcja: Według hipotezy ROME, to właśnie warstwy MLP działają jak pamięć asocjacyjna (Key-Value Memory). To tutaj model „przypomina sobie” fakt. Kiedy sieć widzi wektor reprezentujący „Wieżę Eiffla” (Klucz), warstwa MLP dodaje do strumienia wektor reprezentujący „Paryż” (Wartość).

B. Attention Output (Wyjście uwagi) – Router Informacji
Wzorzec: Często aktywność jest widoczna w ostatnim tokenie całego zapytania (tuż przed predykcją) lub jest bardziej rozproszona.

Funkcja: Mechanizm Attention (uwagi) służy do przenoszenia informacji z jednego miejsca zdania w inne. Gdy warstwa MLP wydobędzie fakt ("Paryż") na pozycji słowa "Tower", mechanizm Attention w późniejszych warstwach "kopiuje" tę informację na koniec zdania ("...is located in [TUTAJ]"), aby model mógł wygenerować słowo.

C. Block Output (Wyjście bloku) – Sumaryczny Efekt
Wzorzec: Jest to suma wyjścia MLP, Attention oraz połączenia rezydualnego (residual connection). Często pokazuje ścieżkę, którą informacja wędruje przez sieć aż do ostatnich warstw.

Funkcja: Pokazuje ogólny przepływ sygnału. Ponieważ jest to suma, często jest mniej precyzyjny w lokalizowaniu samego źródła wiedzy niż czyste mlp_activation, ale pokazuje, jak wiedza narasta w miarę przechodzenia przez warstwy.